# Задание 6.2: Уравнение диффузии-реакции Фишера

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import math
import matplotlib.animation as anim
from scipy.linalg import solve_banded

## Параметры задачи

In [ ]:
L = 1                           # длинна
t_screen = [ 0.01 , 0.1 , 0.2 , 0.5 ] # в какие моменты времени строить графики
dt = 0.01
Nt = max(t_screen)/dt           # кол-во шагов по времени

D = 0.01
r = 1

## Метод FTCS

In [ ]:
def ftcs( u0 , t0 , t1 , dx , dt , D , r ):
    u = u0.copy()
    Nt = math.floor((t1-t0)/dt)
    Nx = len(u0)
    dt_last = t1 - t0 - dt*Nt
    u_new = u.copy()
    for i in range(Nt):
        for i in range(1, Nx - 1):
            u_new[i] = u[i] + dt *( D * (u[i + 1] - 2*u[i] + u[i - 1]) / dx**2 + r * u[i] * (1 - u[i]))
        u_new[0] = u_new[1]
        u_new[-1] = u_new[-2]
        u = u_new.copy()
    if np.fabs(dt_last) > 1.e-8:
        for i in range(1, Nx - 1):
            u_new[i] = u[i] + dt_last *( (D * (u[i + 1] - 2*u[i] + u[i - 1]) / dx**2 + r * u[i] * (1 - u[i])))
        u_new[0] = u_new[1]
        u_new[-1] = u_new[-2]
        u = u_new.copy()
    return u

## Полу-неявный метод 

In [ ]:
def crank_nicolson(u0, t_start, t_end, dt, D, r, dx): 
    u = u0.copy()
    N = len(u) - 1
    cf = (D * dt) / (2 * dx**2)
    Nt = int((t_end - t_start) / dt)

    ab = np.zeros((3, N + 1))
    ab[0, 1:] = -cf
    ab[1, :] = 1 + 2 * cf
    ab[2, :-1] = -cf
    
    ab[0, 1] = -2 * cf
    ab[2, -2] = -2 * cf
    
    for step in range(Nt):
        rhs = np.zeros(N + 1)
        for i in range(1, N):
            rhs[i] = (cf * u[i-1] + 
                     (1 - 2 * cf) * u[i] + 
                     cf * u[i+1] + 
                     dt * r * u[i] * (1 - u[i]))
        rhs[0] = ((1 - 2 * cf) * u[0] + 
                  2 * cf * u[1] + 
                  dt * r * u[0] * (1 - u[0]))
        rhs[N] = (2 * cf * u[N-1] + 
                 (1 - 2 * cf) * u[N] + 
                 dt * r * u[N] * (1 - u[N]))
        u = solve_banded((1, 1), ab, rhs)
    return u

## Вычисления
Рисуются графики для различных параметров Куранта

Видно что при куранте большем 0.5 FTCS разваливается

In [ ]:
t0 = 0   # начальный момент времени
t1 = 1 # конечный момент времени
cfl_screen = [0.4 , 0.5 , 0.51 , 0.52 , 0.53 ,0.6] # какие куранты печатать

CFL = 0.5
Nx = 11                        # кол-во точек по пространству
dx = L/(Nx-1)                   # шаг по пространству
# dt = 0.001341 # шаг по времени
# dt = CFL * dx**2/ D

x = np.linspace(0,L,Nx)        # сетка
u0 = np.exp( -100*(x-0.5)*(x-0.5) ) # нач усл.

for cfl in cfl_screen:
        dt = cfl * dx**2/ D
        u_ftcs = [u0]
    
        u_ftcs = ftcs( u0 , t0 , t1 , dx , dt , D , r )
        u_cn =  crank_nicolson(u0, t0 , t1, dt, D, r, dx)
        
        plt.plot( x , u_ftcs , label=f"FTCS" )
        plt.plot( x , u_cn , label=f"Crank Nicolson" )
        
        plt.xlabel('x, м')
        plt.ylabel('Температура u(x,t)')
        plt.title(f"Сравнение методов в t = {t1} и CFL = {cfl}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

# u_ftcs = [u0]
# t_start = t0
# for i in range(len(t_screen)):
#     u_ftcs.append( ftcs( u_ftcs[i] , t_start , t_screen[i] , dx , dt , D , r ) )
#     t_start = t_screen[i]

# u_cn = [u0]
# t_start = t0
# for i in range(len(t_screen)):
#     u_cn.append( crank_nicolson(u_cn[i], t_start, t_screen[i], dt, D, r, dx) )
#     t_start = t_screen[i]

# for i in range(len(t_screen)):
#     plt.plot( x , u_ftcs[i+1] , label=f"FTCS" )
#     plt.plot( x , u_cn[i+1] , label=f"Crank Nicolson" )
    
#     plt.xlabel('x, м')
#     plt.ylabel('Температура u(x,t)')
#     plt.title(f"Сравнение методов в t = {t_screen[i]}")
#     plt.legend()
#     plt.grid(True, alpha=0.3)
#     plt.show()